In [1]:
# =========================================================
# FINAL MULTIMODAL FUSION SYSTEM
# =========================================================
#
# AUDIO MODEL  : Multiclass Audio Model
# VIDEO MODEL  : EfficientNetB0 Visual Model
#
# AUDIO WEIGHT : 0.75
# VIDEO WEIGHT : 0.25
#
# =========================================================
#
# AUDIO CLASSES:
# hungry
# burping
# belly_pain
# discomfort
# tired
#
# VIDEO CLASSES:
# distressed
# normal
# sleepy
#
# =========================================================
#
# FINAL PIPELINE:
#
# Video
#   ↓
# Extract Audio + Frames
#
# AUDIO:
#   MFCC + Delta + Delta²
#   ↓
#   Audio Prediction
#
# VIDEO:
#   Frame Prediction
#   ↓
#   Probability Averaging
#
# FUSION:
#   Weighted Multimodal Reasoning
#
# =========================================================

# =========================================================
# STEP 1 — INSTALL LIBRARIES
# =========================================================

!pip install -q tensorflow librosa opencv-python

# =========================================================
# STEP 2 — MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive

drive.mount('/content/drive')

# =========================================================
# STEP 3 — IMPORTS
# =========================================================

import os
import cv2
import librosa
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model

from tensorflow.keras.applications.efficientnet import (
    preprocess_input
)

# =========================================================
# STEP 4 — CONFIGURATION
# =========================================================

# ---------------------------------------------------------
# MULTIMODAL WEIGHTS
# ---------------------------------------------------------

AUDIO_WEIGHT = 0.75

VIDEO_WEIGHT = 0.25

# ---------------------------------------------------------
# VIDEO CONFIG
# ---------------------------------------------------------

IMAGE_SIZE = 240

FRAME_SKIP = 10

VIDEO_BATCH_SIZE = 16

# ---------------------------------------------------------
# AUDIO CONFIG
# MUST MATCH TRAINING
# ---------------------------------------------------------

SAMPLE_RATE = 16000

DURATION = 5

N_MFCC = 40

MAX_TIME_STEPS = 100

# =========================================================
# STEP 5 — MODEL PATHS
# =========================================================

AUDIO_MODEL_PATH = (

    "/content/drive/MyDrive/"
    "Audio_Model/"
    "multiclass_cry_model.keras"
)

VIDEO_MODEL_PATH = (

    "/content/drive/MyDrive/"
    "Video_Model/"
    "visual_emotion_model.keras"
)

VIDEO_CLASS_NAMES_PATH = (

    "/content/drive/MyDrive/"
    "Video_Model/"
    "visual_class_names.npy"
)

# =========================================================
# STEP 6 — INPUT VIDEO
# =========================================================

VIDEO_PATH = "/content/drive/MyDrive/fusion_model/final_test.mp4"

# =========================================================
# STEP 7 — AUDIO CLASSES
# =========================================================

AUDIO_CLASSES = [

    "hungry",

    "burping",

    "belly_pain",

    "discomfort",

    "tired"
]

# =========================================================
# STEP 8 — LOAD VIDEO CLASS NAMES
# (Moved to after STEP 9 - CHECK FILES)
# =========================================================

# =========================================================
# STEP 9 — CHECK FILES
# =========================================================

required_files = [

    AUDIO_MODEL_PATH,

    VIDEO_MODEL_PATH,

    VIDEO_CLASS_NAMES_PATH,

    VIDEO_PATH
]

for path in required_files:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\nMissing file:\n{path}"
        )

print("\nAll files found successfully")

# =========================================================
# STEP 8 — LOAD VIDEO CLASS NAMES (moved)
# =========================================================

VIDEO_CLASSES = list(

    np.load(

        VIDEO_CLASS_NAMES_PATH,

        allow_pickle=True
    )
)

# =========================================================
# STEP 10 — LOAD MODELS
# =========================================================

audio_model = load_model(
    AUDIO_MODEL_PATH
)

video_model = load_model(
    VIDEO_MODEL_PATH
)

print("\nModels loaded successfully")

# =========================================================
# STEP 11 — EXTRACT AUDIO
# =========================================================

print("\n===================================")
print("EXTRACTING AUDIO")
print("===================================")

audio_output_path = "/content/temp_audio.wav"

os.system(

    f"ffmpeg -i '{VIDEO_PATH}' "
    f"-q:a 0 -map a "
    f"'{audio_output_path}' "
    f"-y -loglevel quiet"
)

if not os.path.exists(audio_output_path):

    raise ValueError(
        "Audio extraction failed"
    )

print("Audio extracted successfully")

# =========================================================
# STEP 12 — AUDIO FEATURE EXTRACTION
# =========================================================

print("\n===================================")
print("PROCESSING AUDIO")
print("===================================")

audio, sr = librosa.load(

    audio_output_path,

    sr=SAMPLE_RATE
)

# ---------------------------------------------------------
# FIX AUDIO LENGTH
# ---------------------------------------------------------

required_length = SAMPLE_RATE * DURATION

if len(audio) < required_length:

    padding = required_length - len(audio)

    audio = np.pad(
        audio,
        (0, padding)
    )

else:

    audio = audio[:required_length]

# =========================================================
# MFCC
# =========================================================

mfcc = librosa.feature.mfcc(

    y=audio,

    sr=sr,

    n_mfcc=N_MFCC
)

# =========================================================
# DELTA
# =========================================================

delta = librosa.feature.delta(
    mfcc
)

# =========================================================
# DELTA²
# =========================================================

delta2 = librosa.feature.delta(

    mfcc,

    order=2
)

# =========================================================
# STACK FEATURES
# FINAL SHAPE:
# (time_steps, 120)
# =========================================================

features = np.concatenate(

    [mfcc, delta, delta2],

    axis=0
)

features = features.T

# =========================================================
# FIX TIME STEPS
# =========================================================

if features.shape[0] < MAX_TIME_STEPS:

    padding = MAX_TIME_STEPS - features.shape[0]

    features = np.pad(

        features,

        (
            (0, padding),
            (0, 0)
        )
    )

else:

    features = features[:MAX_TIME_STEPS]

# =========================================================
# NORMALIZATION
# MUST MATCH TRAINING
# =========================================================

mean = np.mean(features)

std = np.std(features)

features = (

    features - mean

) / (std + 1e-6)

# =========================================================
# FINAL AUDIO INPUT
# SHAPE:
# (1, 100, 120)
# =========================================================

audio_input = np.expand_dims(

    features,

    axis=0
)

print(f"\nAudio Input Shape: {audio_input.shape}")

# =========================================================
# STEP 13 — AUDIO PREDICTION
# =========================================================

audio_prediction = audio_model.predict(

    audio_input,

    verbose=0
)[0]

audio_index = np.argmax(
    audio_prediction
)

audio_class = AUDIO_CLASSES[
    audio_index
]

audio_confidence = audio_prediction[
    audio_index
]

print("\n===================================")
print("AUDIO PREDICTION")
print("===================================")

print(f"Class      : {audio_class}")

print(f"Confidence : {audio_confidence:.4f}")

# =========================================================
# STEP 14 — VIDEO PROCESSING
# =========================================================

print("\n===================================")
print("PROCESSING VIDEO")
print("===================================")

cap = cv2.VideoCapture(VIDEO_PATH)

frames_batch = []

frame_count = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    if frame_count % FRAME_SKIP != 0:

        continue

    try:

        # -------------------------------------------------
        # BGR → RGB
        # -------------------------------------------------

        frame_rgb = cv2.cvtColor(

            frame,

            cv2.COLOR_BGR2RGB
        )

        # -------------------------------------------------
        # RESIZE
        # -------------------------------------------------

        frame_resized = cv2.resize(

            frame_rgb,

            (
                IMAGE_SIZE,
                IMAGE_SIZE
            )
        )

        # -------------------------------------------------
        # FLOAT ARRAY
        # -------------------------------------------------

        frame_array = np.array(

            frame_resized,

            dtype=np.float32
        )

        # -------------------------------------------------
        # PREPROCESS
        # -------------------------------------------------

        frame_array = preprocess_input(
            frame_array
        )

        frames_batch.append(
            frame_array
        )

    except Exception as e:

        print(
            f"Frame error: {e}"
        )

cap.release()

if len(frames_batch) == 0:

    raise ValueError(
        "No frames extracted"
    )

frames_batch = np.array(
    frames_batch
)

print(

    f"Frames Processed: "
    f"{len(frames_batch)}"
)

# =========================================================
# STEP 15 — VIDEO PREDICTION
# =========================================================

video_predictions = video_model.predict(

    frames_batch,

    batch_size=VIDEO_BATCH_SIZE,

    verbose=1
)

# =========================================================
# FRAME AGGREGATION
# =========================================================

video_probabilities = np.mean(

    video_predictions,

    axis=0
)

video_index = np.argmax(
    video_probabilities
)

video_class = VIDEO_CLASSES[
    video_index
]

video_confidence = video_probabilities[
    video_index
]

print("\n===================================")
print("VIDEO PREDICTION")
print("===================================")

print(f"Class      : {video_class}")

print(f"Confidence : {video_confidence:.4f}")

# =========================================================
# STEP 16 — WEIGHTED FUSION
# =========================================================

print("\n===================================")
print("MULTIMODAL FUSION")
print("===================================")

# ---------------------------------------------------------
# WEIGHTED SCORES
# ---------------------------------------------------------

weighted_audio_score = (

    audio_confidence * AUDIO_WEIGHT
)

weighted_video_score = (

    video_confidence * VIDEO_WEIGHT
)

# =========================================================
# FUSION DECISION
# =========================================================

final_state = ""

reasoning = ""

# ---------------------------------------------------------
# HIGH DISTRESS
# ---------------------------------------------------------

if (

    weighted_audio_score > 0.60

    and

    audio_class in [

        "belly_pain",

        "discomfort"
    ]

    and

    video_class == "distressed"
):

    final_state = "HIGH DISTRESS"

    reasoning = (

        "Strong distress signals "
        "from audio and video"
    )

# ---------------------------------------------------------
# HUNGRY
# ---------------------------------------------------------

elif (

    weighted_audio_score > 0.50

    and

    audio_class == "hungry"
):

    final_state = "HUNGRY"

    reasoning = (

        "Hungry cry pattern "
        "dominates multimodal inference"
    )

# ---------------------------------------------------------
# SLEEPY / TIRED
# ---------------------------------------------------------

elif (

    audio_class == "tired"

    and

    video_class == "sleepy"
):

    final_state = "SLEEPY / TIRED"

    reasoning = (

        "Sleepy visuals and tired cry "
        "detected simultaneously"
    )

# ---------------------------------------------------------
# BURPING
# ---------------------------------------------------------

elif (

    weighted_audio_score > 0.50

    and

    audio_class == "burping"
):

    final_state = "NEEDS BURPING"

    reasoning = (

        "Burping-related cry detected"
    )

# ---------------------------------------------------------
# NORMAL
# ---------------------------------------------------------

elif (

    video_class == "normal"

    and

    video_confidence > 0.70
):

    final_state = "CALM / NORMAL"

    reasoning = (

        "Stable visual emotional state"
    )

# ---------------------------------------------------------
# DEFAULT
# ---------------------------------------------------------

else:

    final_state = (

        f"{audio_class.upper()} + "
        f"{video_class.upper()}"
    )

    reasoning = (
        "Combined multimodal inference"
    )

# =========================================================
# STEP 17 — FINAL CONFIDENCE
# =========================================================

final_confidence = (

    weighted_audio_score

    +

    weighted_video_score
)

# =========================================================
# STEP 18 — FINAL RESULT
# =========================================================

print("\n===================================")
print("FINAL MULTIMODAL RESULT")
print("===================================")

print(f"\nAudio Result : {audio_class}")

print(f"Video Result : {video_class}")

print(f"\nFinal State  : {final_state}")

print(
    f"Confidence   : "
    f"{final_confidence:.4f}"
)

print(f"\nReasoning    : {reasoning}")

# =========================================================
# STEP 19 — AUDIO PROBABILITIES
# =========================================================

print("\n===================================")
print("AUDIO PROBABILITIES")
print("===================================")

for i, class_name in enumerate(AUDIO_CLASSES):

    score = audio_prediction[i]

    bar = "#" * int(score * 30)

    marker = ""

    if i == audio_index:

        marker = " <-- predicted"

    print(

        f"{class_name:<15} "

        f"{score:.4f} "

        f"{bar}"

        f"{marker}"
    )

# =========================================================
# STEP 20 — VIDEO PROBABILITIES
# =========================================================

print("\n===================================")
print("VIDEO PROBABILITIES")
print("===================================")

for i, class_name in enumerate(VIDEO_CLASSES):

    score = video_probabilities[i]

    bar = "#" * int(score * 30)

    marker = ""

    if i == video_index:

        marker = " <-- predicted"

    print(

        f"{class_name:<15} "

        f"{score:.4f} "

        f"{bar}"

        f"{marker}"
    )

# =========================================================
# STEP 21 — COMPLETED
# =========================================================

print("\n===================================")
print("MULTIMODAL ANALYSIS COMPLETED")
print("===================================")

# =========================================================
# STEP 16 — ADVANCED MULTIMODAL FUSION
# =========================================================

print("\n===================================")
print("ADVANCED MULTIMODAL FUSION")
print("===================================")

# ---------------------------------------------------------
# WEIGHTED SCORES
# ---------------------------------------------------------

weighted_audio_score = (

    audio_confidence * AUDIO_WEIGHT
)

weighted_video_score = (

    video_confidence * VIDEO_WEIGHT
)

# ---------------------------------------------------------
# FINAL CONFIDENCE
# ---------------------------------------------------------

final_confidence = (

    weighted_audio_score

    +

    weighted_video_score
)

# =========================================================
# SEMANTIC MULTIMODAL REASONING
# =========================================================

final_state = ""

reasoning = ""

recommendation = ""

# =========================================================
# CASE 1 — HIGH DISTRESS
# =========================================================

if (

    audio_class in [

        "belly_pain",

        "discomfort"
    ]

    and

    video_class == "distressed"

    and

    final_confidence >= 0.65
):

    final_state = (

        "Baby appears to be in "
        "significant distress"
    )

    reasoning = (

        "Pain/discomfort cry pattern "
        "combined with distressed "
        "visual expression"
    )

    recommendation = (

        "Immediate caregiver attention "
        "recommended"
    )

# =========================================================
# CASE 2 — HUNGRY BUT STABLE
# =========================================================

elif (

    audio_class == "hungry"

    and

    video_class == "normal"
):

    final_state = (

        "Baby appears hungry "
        "but emotionally stable"
    )

    reasoning = (

        "Hungry cry pattern detected "
        "without visual distress"
    )

    recommendation = (

        "Feeding may help calm the baby"
    )

# =========================================================
# CASE 3 — SLEEPY / TIRED
# =========================================================

elif (

    audio_class == "tired"

    and

    video_class == "sleepy"
):

    final_state = (

        "Baby appears sleepy and tired"
    )

    reasoning = (

        "Sleep-related cry combined "
        "with drowsy visual cues"
    )

    recommendation = (

        "Rest or sleep may be needed"
    )

# =========================================================
# CASE 4 — NEEDS BURPING
# =========================================================

elif (

    audio_class == "burping"
):

    final_state = (

        "Baby may need burping"
    )

    reasoning = (

        "Burping-related audio "
        "pattern detected"
    )

    recommendation = (

        "Gentle burping posture "
        "may help"
    )

# =========================================================
# CASE 5 — DISTRESS WITHOUT CLEAR AUDIO
# =========================================================

elif (

    video_class == "distressed"

    and

    video_confidence >= 0.70
):

    final_state = (

        "Baby appears visually distressed"
    )

    reasoning = (

        "Strong distressed facial "
        "and emotional cues detected"
    )

    recommendation = (

        "Monitor baby's condition carefully"
    )

# =========================================================
# CASE 6 — CALM / NORMAL
# =========================================================

elif (

    video_class == "normal"

    and

    video_confidence >= 0.70
):

    final_state = (

        "Baby appears calm and stable"
    )

    reasoning = (

        "Normal emotional state "
        "detected visually"
    )

    recommendation = (

        "No immediate concern detected"
    )

# =========================================================
# CASE 7 — DEFAULT FALLBACK
# =========================================================

else:

    final_state = (

        f"Detected state: "
        f"{audio_class} + {video_class}"
    )

    reasoning = (

        "Combined multimodal "
        "prediction generated"
    )

    recommendation = (

        "Continue monitoring "
        "baby behavior"
    )

# =========================================================
# STEP 17 — FINAL RESULT
# =========================================================

print("\n===================================")
print("FINAL MULTIMODAL RESULT")
print("===================================")

print(f"\nAudio Result : {audio_class}")

print(f"Video Result : {video_class}")

print(
    f"\nFinal Interpretation : "
    f"{final_state}"
)

print(
    f"\nConfidence Score : "
    f"{final_confidence:.4f}"
)

print(
    f"\nReasoning : "
    f"{reasoning}"
)

print(
    f"\nRecommendation : "
    f"{recommendation}"
)

Mounted at /content/drive

All files found successfully

Models loaded successfully

EXTRACTING AUDIO
Audio extracted successfully

PROCESSING AUDIO

Audio Input Shape: (1, 100, 120)

AUDIO PREDICTION
Class      : discomfort
Confidence : 0.4051

PROCESSING VIDEO
Frames Processed: 94
6/6 ━━━━━━━━━━━━━━━━━━━━ 64s 9s/step

VIDEO PREDICTION
Class      : distressed
Confidence : 0.4613

MULTIMODAL FUSION

FINAL MULTIMODAL RESULT

Audio Result : discomfort
Video Result : distressed

Final State  : DISCOMFORT + DISTRESSED
Confidence   : 0.4192

Reasoning    : Combined multimodal inference

AUDIO PROBABILITIES
hungry          0.0044 
burping         0.0011 
belly_pain      0.2928 ########
discomfort      0.4051 ############ <-- predicted
tired           0.2965 ########

VIDEO PROBABILITIES
distressed      0.4613 ############# <-- predicted
normal          0.1114 ###
sleepy          0.4274 ############

MULTIMODAL ANALYSIS COMPLETED

ADVANCED MULTIMODAL FUSION

FINAL MULTIMODAL RESULT

Audio Res